# BigEarthNet.txt Evaluation Scores

Inspect synced offline scoring summaries. Model generation and metric computation run on the server; this notebook only compares their exported results.

In [ ]:
import json
from pathlib import Path

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

evaluation_root = repo_root / "outputs/evaluation"
runs = {
    "11283": "no_loc",
    "11284": "loc_text",
    "11285": "loc_embed",
}

def summary_path(job_id: str) -> Path:
    return evaluation_root / job_id / "scored_predictions/summary.json"

available_runs = {job_id: condition for job_id, condition in runs.items() if summary_path(job_id).exists()}
missing_jobs = sorted(set(runs).difference(available_runs))
if missing_jobs:
    print(f"Waiting for synced score summaries: {', '.join(missing_jobs)}")
if not available_runs:
    raise FileNotFoundError(f"No synced score summaries under {evaluation_root}")

summaries = {
    job_id: json.loads(summary_path(job_id).read_text(encoding="utf-8"))
    for job_id in available_runs
}

In [ ]:
task_type_rows = []
for job_id, condition in available_runs.items():
    for row in summaries[job_id]["by_task_type"]:
        task_type_rows.append({"condition": condition, "job_id": job_id, **row})

task_type_scores = pd.DataFrame(task_type_rows).sort_values(["task_type", "condition"])
task_type_scores

In [ ]:
# Positive deltas are improvements over the no-location baseline.
task_type_comparison = task_type_scores.copy()
baseline_by_task = task_type_comparison[task_type_comparison["condition"] == "no_loc"].set_index("task_type")
for metric in ("accuracy", "miou"):
    if metric in task_type_comparison:
        task_type_comparison[f"{metric}_delta_vs_no_loc"] = task_type_comparison.apply(
            lambda row: row[metric] - baseline_by_task.loc[row["task_type"], metric]
            if pd.notna(row.get(metric)) and row["task_type"] in baseline_by_task.index
            else float("nan"),
            axis=1,
        )
task_type_comparison

In [ ]:
caption_rows = [
    {"condition": condition, "job_id": job_id, **summaries[job_id]["captioning"]}
    for job_id, condition in available_runs.items()
]
caption_scores = pd.DataFrame(caption_rows).sort_values("condition")
caption_scores

In [ ]:
caption_comparison = caption_scores.copy()
baseline_caption = caption_comparison.loc[caption_comparison["condition"] == "no_loc"]
if not baseline_caption.empty:
    baseline_caption = baseline_caption.iloc[0]
    for metric in ("bleu1", "bleu2", "bleu3", "bleu4", "meteor", "cider", "rouge_1", "rouge_2", "rouge_l"):
        if metric in caption_comparison:
            caption_comparison[f"{metric}_delta_vs_no_loc"] = caption_comparison[metric] - baseline_caption[metric]
caption_comparison

In [ ]:
task_category_rows = []
for job_id, condition in available_runs.items():
    for row in summaries[job_id]["by_task_category"]:
        task_category_rows.append({"condition": condition, "job_id": job_id, **row})

task_category_scores = pd.DataFrame(task_category_rows).sort_values(
    ["task_type", "task_category", "condition"]
)
task_category_scores

In [ ]:
bounding_box_scores = (
    task_category_scores[task_category_scores["task_type"] == "bounding box"]
    [["condition", "job_id", "task_category", "n", "miou", "acc@25", "acc@50", "acc@75", "acc@90"]]
    .sort_values(["task_category", "condition"])
)
bounding_box_scores